# State of the Data 3: Timeliness
# Notebook 2: Analysis

## Set up the notebook environment

In [1]:
import pandas as pd
import numpy as np
import os, sys
import pyarrow.parquet as pq
import gc
import re
import seaborn as sns
import pickle
import datetime
from datetime import date, timedelta, datetime
from collections import defaultdict

In [2]:
## Create a variable to include in the app informing people when we last ran the Preparation notebook
## For simplicity, we just get the last updated date from the datasette_activities.parquet file
ts = os.path.getmtime('/work/datasette_activities.parquet')
last_run_time = datetime.utcfromtimestamp(ts).strftime('%Y-%m-%d')

## Import data

### Organisations from the organisation Datasette table

In [3]:
table = pq.read_table('datasette_organisations.parquet')
df_ds_organisations = table.to_pandas()
df_ds_organisations.head(5)

In [4]:
df_ds_organisations.shape

In [5]:
df_ds_organisations.info()

In [6]:
df_ds_organisations.describe(include='object')

### Organisations from the Registry

In [7]:
table = pq.read_table('registry_organisation.parquet')
df_reg_organisations = table.to_pandas()
df_reg_organisations.head(5)

In [8]:
df_reg_organisations.info()

In [9]:
df_reg_organisations

In [10]:
df_reg_organisations.shape

In [11]:
df_reg_organisations.describe(include='all')

### Activities from the activity Datasette table

In [12]:
table = pq.read_table('datasette_activities.parquet')
df_ds_activities = table.to_pandas()
df_ds_activities.head(5)

In [13]:
df_ds_activities['hierarchy'] = df_ds_activities['hierarchy'].astype('category')

In [14]:
df_ds_activities.info()

In [15]:
df_ds_activities.shape

In [16]:
print(df_ds_activities.memory_usage(deep=True).sum())

### Transactions from the trans Datasette table

In [17]:
table = pq.read_table('datasette_transaction.parquet')
df_ds_transactions = table.to_pandas()
df_ds_transactions.head(5)

In [18]:
df_ds_transactions.info()

In [19]:
df_ds_transactions.shape

In [20]:
df_ds_transactions.describe(include='all')

### Budgets from the budget Datasette table

In [21]:
table = pq.read_table('datasette_budget.parquet')
df_ds_budget = table.to_pandas()
df_ds_budget.head(5)

In [22]:
df_ds_budget.shape

In [23]:
df_ds_budget.info()

In [24]:
df_ds_budget.describe(include='all')

### Timeliness statistics from the dashboard with warning flags

In [25]:
table = pq.read_table('scraped_timeliness.parquet')
df_scraped_timeliness = table.to_pandas()
df_scraped_timeliness.head(5)

In [26]:
df_scraped_timeliness.info()

In [27]:
df_dash_timeliness = pd.read_csv("timeliness_frequency_with_flags.csv")
df_dash_timeliness.head(5)

In [28]:
df_dash_timeliness.info()

In [29]:
df_dash_timeliness.shape

In [30]:
df_dash_timeliness.describe(include='all')

### Timelag statistics from the dashboard without warning flags

In [31]:
df_dash_timelag = pd.read_csv("timeliness_timelag_without_flags.csv")
df_dash_timelag.head(5)

In [32]:
df_dash_timelag.info()

In [33]:
df_dash_timelag.describe(include='all')

### Current activities from Dashboard comprehensiveness test

In [34]:
df_dashboard_current = pd.read_csv("df_current.csv", index_col=0)

In [35]:
df_dashboard_current

In [36]:
# Create a list of iatiidentifiers that the Dashboard test has labelled as being current
active_list = df_dashboard_current.loc[df_dashboard_current['Active'] == 1]['iatiidentifier'].to_list

### Timeliness data

In [40]:
with open('downloads/most_recent_trans.pkl', 'rb') as f:
    most_recent_trans = pickle.load(f)

In [41]:
#get list of orgs from the dashboard
!wget https://dashboard.iatistandard.org/generated/data/csv/publishers.csv
orgs = pd.read_csv("publishers.csv")
!rm -rf publishers.csv

### All activity dates

In [42]:
act_dates = pd.read_csv("downloads/act_dates.csv")

In [43]:
#download tables
!wget https://iatiregistry.org/publisher/download/csv

#load into dfs
#registry
registry = pd.read_csv("csv")

#remove downloads
!rm -rf "csv"

## Organisation summary data

### Create a new DataFrame, df_summary, for organisation level data
We take our baseline from the organisations that are in the Dashboard, timeliness-frequency table

In [44]:
df_summary = df_scraped_timeliness[['Reporting Org Name','prefix','data-severity']]
df_summary = df_summary.rename({'Reporting Org Name': 'reportingorg_name', 'data-severity': '_flag'}, axis='columns')
df_summary.info()

In [45]:
## Join df_summary on df_reg_organisations
df_summary = pd.merge(df_summary, df_reg_organisations, on='prefix', how='left')
df_summary['organisation_type'] = df_summary['organisation_type'].astype('int')
df_summary

In [46]:
## Download the organisationType codelist from the IATI Standard reference site and create a lookup Dataframe from the type code to its narrative
URL = "https://iatistandard.org/reference_downloads/203/codelists/downloads/clv3/csv/en/OrganisationType.csv"
df_orgtype_codelist  = pd.read_csv(URL)
df_orgtype_codelist = df_orgtype_codelist[["code", "name"]]
df_orgtype_codelist = df_orgtype_codelist.rename(columns={'code': 'organisation_type', 'name': 'Organisation_Type_Name'})
df_orgtype_codelist['organisation_type'] = df_orgtype_codelist['organisation_type'].astype('int')
df_orgtype_codelist

In [47]:
## merge df_orgtype_codelist to add org type narrative to the df_summary DataFrame
df_summary = pd.merge(df_summary, df_orgtype_codelist, on='organisation_type', how='left')
df_summary

In [48]:
# Reorder the columns
df_summary = df_summary.iloc[:, [1, 3, 0, 6, 7, 4, 2, 5 ]]

In [49]:
## Join df_summary on df_dash_timelag
df_summary = pd.merge(df_summary, df_dash_timelag, on='prefix', how='left')

In [50]:
df_summary

In [51]:
## Group activities to get a count per reporting org
df_total_activities = pd.DataFrame(df_ds_activities.groupby('reportingorg_ref').size().sort_values(ascending = False).reset_index())
df_total_activities = df_total_activities.rename({0: 'total_activities'}, axis='columns')
df_total_activities

In [52]:
## merge df_total_activities to add a total activity count to the df_summary DataFrame
df_summary = pd.merge(df_summary, df_total_activities, on='reportingorg_ref', how='left')
df_summary

In [53]:
## Check how many reporting organisations on the dashboard have no published activities
## Flagged as worth investigating further post-SotD
df_summary[df_summary['total_activities'].isna()]

## Analysis

### Dashboard: Timeliness Frequency - weeks

Parse downloaded dictionary into weekly counts

In [54]:
def parse_iso_date(d):
    """Parse a string representation of a date into a datetime object"""
    try:
        return datetime.date(int(d[:4]), int(d[5:7]), int(d[8:10]))
    except (ValueError, TypeError):
        return None

In [55]:
#make dict of all weeks to cover
start_dt = date(2012,1,1)
end_dt = date.today()

#made standard dict for weeks
week_dict = defaultdict(int)
delta = timedelta(days=7)

while start_dt <= end_dt:
    # add current date to list 
    iso_calendar = start_dt.isocalendar()
    week_year = str(iso_calendar[0]) + ", Week " + str(iso_calendar[1])
    week_dict[week_year] = 0
    # increment start date by timedelta
    start_dt += delta

#if current week not included, add to dict
today_week_year = str(end_dt.isocalendar()[0]) + ", Week " + str(end_dt.isocalendar()[1])
if not today_week_year in week_dict.values():
    week_dict[today_week_year] = 0

In [56]:
updates_per_week = {}

for org in most_recent_trans:
    previous_transaction_date = date(1900,1,1)
    updates_per_week[org] = week_dict.copy()
    for gitdate, transaction_date_str in most_recent_trans[org].items():
        if transaction_date_str is not None:
            transaction_date = datetime.strptime(transaction_date_str,'%Y-%m-%d').date() 
            # If transaction date has increased
            if transaction_date > previous_transaction_date:
                #update latest transaction date
                previous_transaction_date = transaction_date
                #pull week of update
                iso_calendar = transaction_date.isocalendar()
                week_year = str(iso_calendar[0]) + ", Week " + str(iso_calendar[1])
                #record update
                updates_per_week[org][week_year] += 1

Make table for past year, add org name from dashboard

In [57]:
week_df = pd.DataFrame()

for org in most_recent_trans:
    updates_per_week[org] = {k:[v] for k,v in updates_per_week[org].items()} 
    df = pd.DataFrame(updates_per_week[org])
    df['regid'] = org
    week_df = pd.concat([week_df, df], ignore_index=True)

In [58]:
#join dashboard table to get org name
week_df_joined = week_df.merge(orgs, how='left', left_on ='regid',right_on='Publisher Registry Id')

#filter to name and 2025 counts
week_df_2025= week_df_joined.filter(regex='Publisher Name|Publisher Registry Id|2025')

#rearrange columns
col = week_df_2025.pop('Publisher Name') 
week_df_2025.insert(0, 'Reporting Organisation', col)  

Add existing frequency measure from dashboard

In [59]:
week_timeliness = week_df_2025.merge(df_scraped_timeliness[['Reporting Org Name', 'prefix','Frequency','First published','data-severity']], how = 'left', left_on = 'Publisher Registry Id', right_on= 'prefix')
week_timeliness = week_timeliness.drop(columns=['Publisher Registry Id','Reporting Org Name'])
week_timeliness = week_timeliness[['prefix'] + [col for col in week_timeliness.columns if col != 'prefix']]
week_timeliness

Calculate if weekly

- For reporting orgs of 6 - 12 months - updates reported in 9 or more of the last 12 weeks (not counting current week).

- For reporting org of 1 year or more - updates reported in 7 or more of the last 12 weeks and at least once in the last 2 weeks (not counting current week).

In [60]:
year_more = ["1-3 years ago","3-5 years ago","More than 5 years ago"]

def weekly_updates(row):
    if row['First published'] == "6-12 months ago":
        if sum(row[-16:-4]) >= 9:
            return("Weekly")
        else: 
            return(row['Frequency'])
    elif row['First published'] in year_more:
        if (sum(row[-16:-4]) >= 7) and (sum(row[-6:-4] >=1)):
            return("Weekly")
        else:
            return(row['Frequency'])
    else:
        return(row['Frequency'])

In [61]:
week_timeliness['Updated Frequency'] = week_timeliness.apply(weekly_updates, axis=1)
week_timeliness 

In [62]:
## Add the Frequency and Updated Frequency columns to df_summary, rename
df_summary = df_summary.merge(week_timeliness[['prefix', 'Frequency','Updated Frequency']], how = 'left', left_on = 'prefix', right_on= 'prefix')
## df_summary.rename(columns={'Frequency': 'Frequency Classic', 'Updated Frequency': 'Frequency'}, inplace=True)
df_summary.info()

In [63]:
# Get the reportingorg_names as a list for organizations with Weekly frequency
weekly_orgs_list = df_summary.loc[df_summary['Updated Frequency'] == "Weekly", 'reportingorg_name'].tolist()
weekly_orgs_list

### Dashboard: Timeliness Frequency

In [64]:
## To create a stacked bar chart in the Streamlit application, we group by both Organisation_Type_Name and Frequency
df_timeliness_by_orgtype = pd.DataFrame(df_summary.groupby(['Organisation_Type_Name', 'Frequency']).size().reset_index())
df_timeliness_by_orgtype = df_timeliness_by_orgtype.rename(columns={0: "Number of Organisations"})
df_timeliness_by_orgtype

In [65]:
## To create a stacked bar chart in the Streamlit application, we group by both Organisation_Type_Name and Frequency
df_updated_timeliness_by_orgtype = pd.DataFrame(df_summary.groupby(['Organisation_Type_Name', 'Updated Frequency']).size().reset_index())
df_updated_timeliness_by_orgtype = df_updated_timeliness_by_orgtype.rename(columns={0: "Number of Organisations"})
df_updated_timeliness_by_orgtype

In [66]:
## To create a stacked bar chart in the Streamlit application, we group by both Organisation_Type_Name and Frequency, counting up the total number of activities per organisation
df_timeliness_aggregate_activities = pd.DataFrame(df_summary.groupby(['Organisation_Type_Name', 'Frequency'])['total_activities'].sum().reset_index())
df_timeliness_aggregate_activities = df_timeliness_aggregate_activities.rename(columns={"total_activities": "Number of Activities"})
df_timeliness_aggregate_activities

In [67]:
## To create a stacked bar chart in the Streamlit application, we group by both Organisation_Type_Name and Frequency, counting up the total number of activities per organisation
df_updated_timeliness_aggregate_activities = pd.DataFrame(df_summary.groupby(['Organisation_Type_Name', 'Updated Frequency'])['total_activities'].sum().reset_index())
df_updated_timeliness_aggregate_activities = df_updated_timeliness_aggregate_activities.rename(columns={"total_activities": "Number of Activities"})
df_updated_timeliness_aggregate_activities

### Dashboard: Timeliness Timelag

In [68]:
## To create a stacked bar chart in the Streamlit application, we group by both Organisation_Type_Name and Frequency
df_timelag_by_orgtype = pd.DataFrame(df_summary.groupby(['Organisation_Type_Name', 'Time lag']).size().reset_index())
df_timelag_by_orgtype = df_timelag_by_orgtype.rename(columns={0: "Number of Organisations"})
df_timelag_by_orgtype

In [69]:
## To create a stacked bar chart in the Streamlit application, we group by both Organisation_Type_Name and Frequency, counting up the total number of activities per organisation
df_timelag_aggregate_activities = pd.DataFrame(df_summary.groupby(['Organisation_Type_Name', 'Time lag'])['total_activities'].sum().reset_index())
df_timelag_aggregate_activities = df_timelag_aggregate_activities.rename(columns={"total_activities": "Number of Activities"})
df_timelag_aggregate_activities

### Activity: Activity Status

Here, we briefly investigate duplicated activities before dropping the duplicates. 
Then we group by activity-status to produce a table of statuses and the number of activities in them.
Next, we group by both activity-status and organisation-type.
Then we identify 'current' activities by two methods:
- activity-status = 2 (Implementation)
- Dashboard's comprehensiveness test for current activities
activity-status is in "real time" whereas the Dashboard has a tolerance of a year.

In [70]:
## Check how many unique activites we have, based on 'iatiidentifier'
print(df_ds_activities.shape)
unique_activities = (df_ds_activities['iatiidentifier'].nunique())
print(unique_activities)
unique_activities_orgs = df_ds_activities['reportingorg_ref'].nunique()
print (unique_activities_orgs)

In [71]:
## Create a DataFrame of duplicated iatiidentifiers
df_dup_activities = df_ds_activities[df_ds_activities.duplicated('iatiidentifier', keep='first')]
df_dup_activities.info()

In [72]:
unique_dupes = (df_dup_activities['iatiidentifier'].nunique())
print(unique_dupes)

In [73]:
## Group on prefix to identify the organisations publishing duplicate iatiidentifiers
## While useful to know this, it is outside the scope of timely data so we don't merge this into df_summary
df_dup_activities.groupby('prefix').size().sort_values(ascending = False).reset_index()

In [74]:
## We know that SDC_CH publish the same files in multiple datasets. 
## Drop duplicates from the df_ds_activities DataFrame
df_ds_activities = df_ds_activities.drop_duplicates(subset=['iatiidentifier'])
df_ds_activities

In [75]:
## Count the number of unique activities and the number of reporting organisations of activities
print(df_ds_activities.shape)
unique_activities = (df_ds_activities['iatiidentifier'].nunique())
print(unique_activities)
unique_activities_orgs = df_ds_activities['reportingorg_ref'].nunique()
print (unique_activities_orgs)

In [76]:
## Group by activity status to give us a high-level (Big Numbers) snapshot of the status of activities.
# df_ds_activities.groupby(['activitystatus_codename']).size().sort_values(ascending = False).reset_index()
df_grouped_activities = df_ds_activities.groupby(['activitystatus_codename']).size().sort_values(ascending = False).reset_index(name='Number of Activities')
df_grouped_activities = df_grouped_activities.rename(columns={'activitystatus_codename': 'Activity Status'})
df_grouped_activities

In [77]:
## To create a stacked bar chart in the Streamlit application, we group by both Organisation_Type_Name and Frequency, counting up the total number of activities per organisation
df_grouped_stacked_activities = pd.DataFrame(df_ds_activities.groupby(['activitystatus_codename','reportingorg_type']).size().reset_index(name = "Number of Activities"))
df_grouped_stacked_activities = df_grouped_stacked_activities.rename(columns={"reportingorg_type": "organisation_type", 'activitystatus_codename': 'Activity Status'})
df_grouped_stacked_activities 

In [78]:
# Convert to organisation_type to numeric, handling any non-numeric values by coercing them to NaN
df_grouped_stacked_activities['organisation_type'] = pd.to_numeric(df_grouped_stacked_activities['organisation_type'], errors='coerce')
# Convert to Int64 (nullable integer type) to preserve NaN values
df_grouped_stacked_activities['organisation_type'] = df_grouped_stacked_activities['organisation_type'].astype('Int64')
df_grouped_stacked_activities

In [79]:
## Now drop the rows with NaN values for organisation type
df_grouped_stacked_activities = df_grouped_stacked_activities.loc[df_grouped_stacked_activities['organisation_type'].notna()]

In [80]:
## Merge in Organisation_Type_Name
df_grouped_stacked_activities = pd.merge(df_grouped_stacked_activities, df_orgtype_codelist, on='organisation_type', how='left')
df_grouped_stacked_activities

#### Use activitystatus codename of "Implementation' in the activities data to determine current activities

In [81]:
## Create a new DataFrame of activities that have an activitystatus of "Implementation"
df_implementing_activities = df_ds_activities.loc[df_ds_activities['activitystatus_codename']=="Implementation"].reset_index(drop=True)
df_implementing_activities

In [82]:
## Count the number of unique activities and the number of reporting organisations of activities with an activity-status of Implementation
unique_implementing_activities = df_implementing_activities['iatiidentifier'].nunique()
print (unique_implementing_activities)
unique_implementing_orgs = df_implementing_activities['reportingorg_ref'].nunique()
print (unique_implementing_orgs)

#### Use activity status codename of 'Implementation' plus actual end date from activity data to determine current activities

In [83]:
## Create a DataFrame of activities in Implementation status and with no end date
df_implementing_activities_noenddate = df_ds_activities.loc[(df_ds_activities['actualend'].isna()) & (df_ds_activities['activitystatus_codename'] == 'Implementation')].reset_index(drop=True)
df_implementing_activities_noenddate

In [84]:
unique_implementing_activities_noenddate = df_implementing_activities_noenddate['iatiidentifier'].nunique()
print (unique_implementing_activities_noenddate)
unique_implementing_orgs_noenddate = df_implementing_activities_noenddate['reportingorg_ref'].nunique()
print (unique_implementing_orgs_noenddate)

#### Use Dashboard comprehensiveness test for current activities

In [85]:
df_dashboard_current_activities = pd.DataFrame(df_ds_activities.loc[df_ds_activities['iatiidentifier'].isin(active_list())].reset_index(drop=True))
df_dashboard_current_activities

In [86]:
unique_implementing_activities_dashboard = df_dashboard_current_activities['iatiidentifier'].nunique()
print (unique_implementing_activities_dashboard)
unique_implementing_orgs_dashboard = df_dashboard_current_activities['reportingorg_ref'].nunique()
print (unique_implementing_orgs_dashboard)

### Activity: Interaction between activity-status and end dates

#### In Implementation, but has an actual end date

In [87]:
## Create a Dataframe of activities in Implementation but that have an actual end date
df_implementation_ended = df_ds_activities.loc[(df_ds_activities['activitystatus_codename'] == 'Implementation') & (df_ds_activities['actualend'].notna())]
df_implementation_ended = df_implementation_ended.drop_duplicates(subset=['iatiidentifier'])
unique_implementation_ended = df_implementation_ended.shape[0]

In [88]:
df_implementation_ended

In [ ]:
## list all of the activities with an activity status of "Implementation" but have an actual end date in the past
df_ds_activities.loc[(df_ds_activities['activitystatus_codename'] == 'Implementation') & 
                           (df_ds_activities['actualend'].notna()) & 
                           (df_ds_activities['actualend'] < pd.Timestamp.now(tz='UTC'))]

In [89]:
## Group the activities in Implementation but with an actual end date by year of that actual end date
df_implementation_ended_byyear = df_implementation_ended.groupby(df_implementation_ended['actualend'].dt.to_period('Y')).size().sort_values(ascending = False).reset_index(name='Number of Activities')
df_implementation_ended_byyear['actualend'] = df_implementation_ended_byyear['actualend'].astype(str)
df_implementation_ended_byyear = df_implementation_ended_byyear[df_implementation_ended_byyear.actualend != '1970']
df_implementation_ended_byyear = df_implementation_ended_byyear.rename(columns={'actualend': 'Actual End Date'})

In [90]:
df_implementation_ended_byyear

In [ ]:
df_implementation_ended.groupby('reportingorg_ref').size().sort_values(ascending = False).reset_index()

#### Is Closed but has no actual end date

In [91]:
df_closed_activities = df_ds_activities.loc[df_ds_activities['activitystatus_codename']=="Closed"].reset_index(drop=True)
df_closed_activities

In [92]:
df_closed_notended = df_ds_activities.loc[(df_ds_activities['activitystatus_codename'] == 'Closed') & (df_ds_activities['actualend'].isna())]
df_closed_notended = df_closed_notended.drop_duplicates(subset=['iatiidentifier'])
unique_closed_notended = df_closed_notended.shape[0]
print(unique_closed_notended)

In [93]:
## Remove the rows with no planned end date
df_closed_notended = df_closed_notended[(df_closed_notended['plannedend'].notna())]
unique_closed_notended = df_closed_notended.shape[0]
print(unique_closed_notended)

In [94]:
df_closed_notended_byyear = df_closed_notended.groupby(df_closed_notended['plannedend'].dt.to_period('Y')).size().sort_values(ascending = False).reset_index(name='Number of Activities')
df_closed_notended_byyear['plannedend'] = df_closed_notended_byyear['plannedend'].astype(str)
df_closed_notended_byyear['plannedend'] = df_closed_notended_byyear['plannedend'].astype(int)
## df_closed_notended_byyear = df_closed_notended_byyear[df_implementation_ended_byyear.actualend != '1970']
df_closed_notended_byyear = df_closed_notended_byyear.rename(columns={'plannedend': 'Planned End Date'})
df_closed_notended_byyear

In [95]:
df_closed_notended_byyear = df_closed_notended_byyear.loc[(df_closed_notended_byyear['Planned End Date'] >= 2000) & (df_closed_notended_byyear['Planned End Date'] <= 2030)]
df_closed_notended_byyear

#### Look at the many activities in Finalisation

In [96]:
## Create a DataFrame, df_finalisation, froom the activities with an activity-status of Finalisation
df_finalisation = df_ds_activities.loc[(df_ds_activities['activitystatus_codename'] == 'Finalisation')]
df_finalisation = df_finalisation.drop_duplicates(subset=['iatiidentifier'])
unique_finalisation = df_finalisation.shape[0]
print(unique_finalisation)

In [97]:
df_finalisation.info()

In [98]:
## Group on reportingorg_ref to see how many activities are in Finalisation per organisation
df_finalisation_orgs = pd.DataFrame(df_finalisation.groupby('reportingorg_ref').size().sort_values(ascending = False).reset_index())
df_finalisation_orgs

In [99]:
## Split df_finalised depending on whether there is an actual end date or not
df_finalisation_actualend = df_finalisation[(df_finalisation['actualend'].notna())]
df_finalisation_notended = df_finalisation[(df_finalisation['actualend'].isna())]
finalisation_notended = df_finalisation_notended.shape[0]
print(finalisation_notended)
finalisation_actualend = df_finalisation_actualend.shape[0]
print (finalisation_actualend)

In [100]:
df_finalisation_actualend

In [101]:
df_finalisation_actualend_byyear = df_finalisation_actualend.groupby(df_finalisation_actualend['actualend'].dt.to_period('Y')).size().sort_values(ascending = False).reset_index(name='Number of Activities')
df_finalisation_actualend_byyear['actualend'] = df_finalisation_actualend_byyear['actualend'].astype(str)
df_finalisation_actualend_byyear['actualend'] = df_finalisation_actualend_byyear['actualend'].astype(int)
df_finalisation_actualend_byyear = df_finalisation_actualend_byyear.loc[(df_finalisation_actualend_byyear['actualend'] >= 2000) & (df_finalisation_actualend_byyear['actualend'] <= 2030)]
df_finalisation_actualend_byyear = df_finalisation_actualend_byyear.rename(columns={'actualend': 'Actual End Date'})
df_finalisation_actualend_byyear

In [102]:
_dntk.DeepnoteChart(df_finalisation_actualend_byyear, """{"layer":[{"layer":[{"layer":[{"mark":{"clip":true,"type":"bar","color":"#2266D3","tooltip":true},"encoding":{"x":{"axis":{"grid":false},"sort":"ascending","type":"temporal","field":"Actual End Date","scale":{"type":"linear"},"timeUnit":"year","bandPosition":0},"y":{"axis":{"format":{"type":"default","decimals":null},"formatType":"numberFormatFromNumberType"},"type":"quantitative","field":"Number of Activities","scale":{"type":"linear"},"format":{"type":"default","decimals":null},"aggregate":"sum","formatType":"numberFormatFromNumberType"},"color":{"type":"nominal","datum":"Series","scale":{"range":["#2266D3"],"domain":["Series"]}},"xOffset":{"datum":"series_0"}},"transform":[]}]}],"resolve":{"scale":{"color":"independent"}}}],"title":"","config":{"legend":{"disable":false}},"$schema":"https://vega.github.io/schema/vega-lite/v5.json","encoding":{},"usermeta":{"seriesNames":["Series"],"seriesOrder":[0],"specSchemaVersion":2,"tooltipDefaultMode":true}}""", attach_selection=True, filters='[]')

In [103]:
df_finalisation_notended

In [104]:
_dntk.DeepnoteChart(df_finalisation_notended, """{"layer":[{"layer":[{"layer":[{"mark":{"clip":true,"type":"bar","color":"#2266D3","tooltip":true},"encoding":{"x":{"axis":{"grid":false},"sort":"ascending","type":"temporal","field":"plannedend","scale":{"type":"linear"},"timeUnit":"year","bandPosition":0},"y":{"axis":{"format":{"type":"default","decimals":null},"formatType":"numberFormatFromNumberType"},"type":"quantitative","scale":{"type":"linear"},"format":{"type":"default","decimals":null},"aggregate":"count","formatType":"numberFormatFromNumberType"},"color":{"type":"nominal","datum":"Series","scale":{"range":["#2266D3"],"domain":["Series"]}},"xOffset":{"datum":"series_0"}},"transform":[]}]}],"resolve":{"scale":{"color":"independent"}}}],"title":"","config":{"legend":{"disable":false}},"$schema":"https://vega.github.io/schema/vega-lite/v5.json","encoding":{},"usermeta":{"seriesNames":["Series"],"seriesOrder":[0],"specSchemaVersion":2,"tooltipDefaultMode":true}}""", attach_selection=True, filters='[]')

In [105]:
df_finalisation_notended_byyear = df_finalisation_notended.groupby(df_finalisation_notended['plannedend'].dt.to_period('Y')).size().sort_values(ascending = False).reset_index(name='Number of Activities')
df_finalisation_notended_byyear['plannedend'] = df_finalisation_notended_byyear['plannedend'].astype(str)
df_finalisation_notended_byyear['plannedend'] = df_finalisation_notended_byyear['plannedend'].astype(int)
df_finalisation_notended_byyear = df_finalisation_notended_byyear.loc[(df_finalisation_notended_byyear['plannedend'] >= 2000) & (df_finalisation_notended_byyear['plannedend'] <= 2030)]
df_finalisation_notended_byyear = df_finalisation_notended_byyear.rename(columns={'plannedend': 'Planned End Date'})
df_finalisation_notended_byyear

In [106]:
_dntk.DeepnoteChart(df_finalisation_notended_byyear, """{"layer":[{"layer":[{"layer":[{"mark":{"clip":true,"type":"bar","color":"#2266D3","tooltip":true},"encoding":{"x":{"axis":{"grid":false},"sort":"ascending","type":"temporal","field":"Planned End Date","scale":{"type":"linear"},"timeUnit":"year","bandPosition":0},"y":{"axis":{"format":{"type":"default","decimals":null},"formatType":"numberFormatFromNumberType"},"type":"quantitative","field":"Number of Activities","scale":{"type":"linear"},"format":{"type":"default","decimals":null},"formatType":"numberFormatFromNumberType"},"color":{"type":"nominal","datum":"Number of Activities","scale":{"range":["#2266D3"],"domain":["Number of Activities"]}},"xOffset":{"datum":"series_0"}},"transform":[]}]}],"resolve":{"scale":{"color":"independent"}}}],"title":"","config":{"legend":{"disable":false}},"$schema":"https://vega.github.io/schema/vega-lite/v5.json","encoding":{},"usermeta":{"seriesNames":["Number of Activities"],"seriesOrder":[0],"specSchemaVersion":2,"tooltipDefaultMode":true}}""", attach_selection=True, filters='[]')

In [107]:
## Create some variables to hold summary stats about organisations and how many activities they report
mean_activities = (int(df_summary['total_activities'].mean(numeric_only=True)))
median_activities = (int(df_summary['total_activities'].median(numeric_only=True)))
mode_activities = (int(df_summary['total_activities'].mode()))
max_activities = (int(df_summary['total_activities'].max()))
org_with_max_activities = (df_summary.loc[df_summary['total_activities'].idxmax(), 'reportingorg_name'])

### Activity: Activity dates

Start/end date plots. Preparing the data:

In [108]:
#remove activities with multiple dates of the same type
dup_dates = act_dates[act_dates.duplicated(['reportingorg_ref','iatiidentifier','typename'])]
act_dates = act_dates[~act_dates.iatiidentifier.isin(dup_dates['iatiidentifier'])]

#pivot to one line per activity
activity_dates_pivot = act_dates.pivot(index=['reportingorg_ref','iatiidentifier'], columns='typename').reset_index()
activity_dates_pivot.columns = activity_dates_pivot.columns.to_series().apply(''.join)

#reorder columns
activity_dates_pivot = activity_dates_pivot[['reportingorg_ref','iatiidentifier', 'isodatePlanned start','isodateActual start','isodatePlanned End','isodateActual end']]

In [109]:
## Count the total number of activity dates
unique_act_dates = act_dates.shape[0]
print(unique_act_dates)


Join on reporting org type from registry - drop if doesn't match

In [110]:
activity_dates_joined = activity_dates_pivot.merge(registry[['IATI Organisation Identifier','Organization Type']],
    how = 'inner', left_on = 'reportingorg_ref', right_on = 'IATI Organisation Identifier')
activity_dates_joined

### Last updated date times

In [111]:
## Note that outliers will drag the mean down here and so the min, 25%, 50%, 75% and max values are more instructive
df_ds_activities['lastupdateddatetime'].describe()

In [112]:
_dntk.DeepnoteChart(df_ds_activities, """{"layer":[{"layer":[{"layer":[{"mark":{"clip":true,"type":"bar","color":"#2266D3","tooltip":true},"encoding":{"x":{"axis":{"grid":false},"sort":"ascending","type":"temporal","field":"lastupdateddatetime","scale":{"type":"linear"},"timeUnit":"yearmonth","bandPosition":0},"y":{"axis":{"format":{"type":"default","decimals":null},"formatType":"numberFormatFromNumberType"},"type":"quantitative","scale":{"type":"linear"},"format":{"type":"default","decimals":null},"aggregate":"count","formatType":"numberFormatFromNumberType"},"color":{"type":"nominal","datum":"Series","scale":{"range":["#2266D3"],"domain":["Series"]}},"xOffset":{"datum":"series_0"}},"transform":[]}]}],"resolve":{"scale":{"color":"independent"}}}],"title":"","config":{"legend":{"disable":false}},"$schema":"https://vega.github.io/schema/vega-lite/v5.json","encoding":{},"usermeta":{"seriesNames":["Series"],"seriesOrder":[0],"specSchemaVersion":2,"tooltipDefaultMode":true}}""", attach_selection=True, filters='[]')

#### Investigate the activities with null values for lastupdateddatetime

In [113]:
## Create a new DataFrame from the rows where there is no lastupdateddatetime
df_lastupdated_nulls = df_ds_activities[df_ds_activities['lastupdateddatetime'].isnull()]
df_lastupdated_nulls.info()


In [114]:
## Group on reportingorg_ref to see which organisations are publishing the most null laastupdateddatetime values
## What proportion of their activities does this represent?
df_lastupdated_nulls.groupby('reportingorg_ref').size().sort_values(ascending = False).reset_index()

In [115]:
## Create a new DataFrame df_lastupdated by removing the rows with null lastupdateddatetimevalues from the df_ds_activities DataFrame
df_lastupdated = df_ds_activities.dropna(subset=['lastupdateddatetime'])
df_lastupdated.info()

In [116]:
## Save the number of nulls and non-nulls as variables
lastupdated_nulls = df_lastupdated_nulls.shape[0]
print(lastupdated_nulls)
lastupdated_nonnulls = df_lastupdated.shape[0]
print(lastupdated_nonnulls)

In [117]:
## Create separate 'year' and 'month' columns from lastupdateddatetime
df_lastupdated["Month"] = df_lastupdated["lastupdateddatetime"].dt.month_name()
df_lastupdated["Year"] = df_lastupdated["lastupdateddatetime"].dt.year

In [118]:
df_lastupdated.head(5)

#### Dates from before the first recorded IATI publication
There are some activities with last updated dates from before the first IATI publication in January 2011 (taken from https://iatistandard.org/en/about/iati-history/). Create a dataframe too_old for any activities with a year before 2011.

In [119]:
df_too_old=df_lastupdated.loc[df_lastupdated['Year'] <= 2010]  


In [120]:
df_too_old.info()

In [121]:
lastupdated_old = df_too_old.shape[0]
print(lastupdated_old)

#### Group and order the organisations that publish the most activities with pre-2011 dates

In [122]:
## ## Group on reportingorg_ref to see which organisations are publishing the old lastupdateddatetime values
df_grouped_too_old = df_too_old.groupby('reportingorg_ref').size().sort_values(ascending = False).reset_index()
df_grouped_too_old

In [123]:
## Drop the rows with lastupdateddatetime before 2011 as this was before IATI's first publication
df_lastupdated = df_lastupdated.loc[df_lastupdated['Year'] >= 2011]  
lastupdated_iati_era = df_lastupdated.shape[0]
print(lastupdated_iati_era)

In [124]:
df_lastupdated_byyear = pd.DataFrame(df_lastupdated.groupby(pd.Grouper(key='lastupdateddatetime', axis=0, freq='Y')).size().sort_values(ascending = False).reset_index())
df_lastupdated_byyear = df_lastupdated_byyear.rename(columns={0: "count"})
df_lastupdated_byyear['lastupdateddatetime'] = pd.DatetimeIndex(df_lastupdated_byyear['lastupdateddatetime']).year
df_lastupdated_byyear

In [125]:
# Calculate number of activities where lastupdateddatetime is before 2024
lastupdated_acts_before_2024 = df_lastupdated_byyear[df_lastupdated_byyear['lastupdateddatetime'] < 2024]['count'].sum()
print(lastupdated_acts_before_2024)

In [126]:
## Create a DataFrame with a row for each reporting org and the date of their earliest lastupdateddatetime
df_first_updatedate = pd.DataFrame(df_lastupdated.groupby('reportingorg_ref')['lastupdateddatetime'].min().reset_index())
df_first_updatedate = df_first_updatedate.rename(columns={"lastupdateddatetime": "min_lastupdateddatetime"})
df_first_updatedate

In [127]:
## Create a DataFrame with a row for each reporting org and the date of their latest lastupdateddatetime
df_last_updateddate = pd.DataFrame(df_lastupdated.groupby('reportingorg_ref')['lastupdateddatetime'].max().reset_index())
df_last_updateddate = df_last_updateddate.rename(columns={"lastupdateddatetime": "max_lastupdateddatetime"})
df_last_updateddate

In [128]:
## Merge the two DataFrames into df_summary
df_summary = pd.merge(df_summary, df_first_updatedate, on='reportingorg_ref', how='left')
df_summary

In [129]:
df_summary = pd.merge(df_summary, df_last_updateddate, on='reportingorg_ref', how='left')
df_summary

In [130]:
## Create a new column to contain the difference between the earliest and latest latestupdateddatetime
df_summary['difflastfirstupdateddate'] = df_summary['max_lastupdateddatetime'] - df_summary['min_lastupdateddatetime']
df_summary['difflastfirstupdateddate'] = df_summary['difflastfirstupdateddate'].dt.days.astype(float)
df_summary.sort_values(['difflastfirstupdateddate', 'total_activities'], ascending=[True, False])

In [131]:
## Save the number of organisation that have no difference between their earliest and latest lastupdateddatetime and count their aggregate activities
zerodifforgs = df_summary[df_summary.difflastfirstupdateddate == 0].shape[0]
print(zerodifforgs)
zerodiffactivities = df_summary.loc[df_summary['difflastfirstupdateddate'] == 0, 'total_activities'].sum().astype(int)
print(zerodiffactivities)

In [132]:
df = df_summary[df_summary['total_activities'].notna()]
df

In [133]:
df['difflastfirstupdateddate'] = df['difflastfirstupdateddate']/365

In [134]:
## Create a DataFrame showing the distribution of differences between first and last update dates
df_lastupdateddiff = pd.DataFrame(df.groupby('difflastfirstupdateddate').size().reset_index())
df_lastupdateddiff = df_lastupdateddiff.rename(columns={0: "Number of Organisations"})
df_lastupdateddiff

In [135]:
_dntk.DeepnoteChart(df_lastupdateddiff, """{"layer":[{"layer":[{"layer":[{"mark":{"clip":true,"type":"bar","color":"#2266D3","tooltip":true},"encoding":{"x":{"bin":{"step":1},"sort":"ascending","type":"quantitative","field":"difflastfirstupdateddate","scale":{"type":"linear"}},"y":{"axis":{"format":{"type":"default","decimals":null},"formatType":"numberFormatFromNumberType"},"type":"quantitative","field":"Number of Organisations","scale":{"type":"linear"},"format":{"type":"default","decimals":null},"aggregate":"sum","formatType":"numberFormatFromNumberType"},"color":{"type":"nominal","datum":"Number of Organisations","scale":{"range":["#2266D3"],"domain":["Number of Organisations"]}}},"transform":[]}]}],"resolve":{"scale":{"color":"independent"}}}],"title":"","config":{"legend":{"disable":false}},"$schema":"https://vega.github.io/schema/vega-lite/v5.json","encoding":{},"usermeta":{"seriesNames":["Number of Organisations"],"seriesOrder":[0],"specSchemaVersion":2,"tooltipDefaultMode":true}}""", attach_selection=True, filters='[]')

In [136]:
df_lastupdated_orgs_byyear = pd.DataFrame(df_lastupdated.groupby(['reportingorg_ref', 'Year']).size().sort_values(ascending = False).reset_index())
df_lastupdated_orgs_byyear = df_lastupdated_orgs_byyear.rename(columns={0: "Number of Activities"})
df_lastupdated_orgs_byyear

### Schema Rules
There are seven date/time related rules that we can test. 
11.1.1: The last updated datetime of the activity must not be in the future.
11.1.2: The planned start date of the activity must be before the planned end date.
11.1.3: The actual start date of the activity must be before the actual end date.
11.1.4: The actual start date of the activity must not be in the future.
11.1.5: The actual end date of the activity must not be in the future.
11.2.1: The transaction date must not be in the future.
11.2.2: The transaction value date must not be in the future.

#### 11.1.1 The last updated datetime of the activity must not be in the future.

In [139]:
# List the activities with a lastupdateddatetime in the future
df_future_lastupdated = pd.DataFrame((df_ds_activities.loc[df_ds_activities['lastupdateddatetime'] >= pd.Timestamp.now(tz='UTC')]))
df_future_lastupdated

#### 11.1.2 The planned start date of the activity must be before the planned end date.

In [141]:
# First, note the rows returned when we use greater than or equal to - do we want to exclude 'cancelled' activities?
df_ds_activities.loc[df_ds_activities['plannedstart'] >= df_ds_activities['plannedend']]

In [142]:
df_end_before_start_planned = pd.DataFrame(df_ds_activities.loc[df_ds_activities['plannedstart'] > df_ds_activities['plannedend']])
df_end_before_start_planned

In [143]:
df_end_before_start_planned['ruleset']='11.1.2'
df_end_before_start_planned

#### 11.1.3: The actual start date of the activity must be before the actual end date.

In [145]:
# List the activities where the actual start date is equal to or after the actual end date
pd.DataFrame(df_ds_activities.loc[df_ds_activities['actualstart'] >= df_ds_activities['actualend']])

In [146]:
# List the activities where the actual start date is after the actual end date
df_end_before_start_actual = pd.DataFrame(df_ds_activities.loc[df_ds_activities['actualstart'] > df_ds_activities['actualend']])
df_end_before_start_actual

In [147]:
df_end_before_start_actual['ruleset']='11.1.3'
df_end_before_start_actual

#### 11.1.4: The actual start date of the activity must not be in the future.

In [149]:
# List the organisations with an actual start date after today
df_future_start = pd.DataFrame((df_ds_activities.loc[df_ds_activities['actualstart'] >= pd.Timestamp.now(tz='UTC')]))
df_future_start

In [150]:
df_future_start['ruleset']='11.1.4'
df_future_start

In [151]:
# List the activities with an actual start date after their lastupdateddatetime
df_ds_activities.loc[df_ds_activities['actualstart'] >= df_ds_activities['lastupdateddatetime']]

#### 11.1.5: The actual end date of the activity must not be in the future.

In [153]:
# List the organisations with an actual end date after today
df_future_end = pd.DataFrame((df_ds_activities.loc[df_ds_activities['actualend'] > pd.Timestamp.now(tz='UTC')]))
df_future_end
# df.groupby(['reportingorg_ref']).size()

In [154]:
df_future_end['ruleset']='11.1.5'
df_future_end

In [155]:
# List the activities with an actual end date in a future year
df_ds_activities.loc[df_ds_activities['actualend'] >= pd.Timestamp(2026, 1, 1, tz='UTC')]

In [156]:
# List the activities with an actual end date after their lastupdateddatetime
df_ds_activities.loc[df_ds_activities['actualend'] > df_ds_activities['lastupdateddatetime']]

#### 11.2.1: The transaction date must not be in the future.


In [159]:
# Create a DataFrame, df_future_trans, with the transactions that have a 'transactiondate_isodate' in the future
df_future_trans = pd.DataFrame((df_ds_transactions.loc[df_ds_transactions['transactiondate_isodate'] >= pd.Timestamp.now(tz='UTC')]))
df_future_trans

In [160]:
df_future_trans['ruleset']='11.2.1'
df_future_trans

In [161]:
# List the transactions with a 'transactiondate_isodate' in a future year
df_ds_transactions.loc[df_ds_transactions['transactiondate_isodate'] >= pd.Timestamp(2026, 1, 1, tz='UTC')]

#### 11.2.2: The transaction value date must not be in the future.

In [163]:
# Create a DataFrame, df_future_value, with the transactions that have a 'value_valuedate' in the future
# Fix: Make sure both sides of the comparison have compatible timezone information
df_future_value = pd.DataFrame((df_ds_transactions.loc[df_ds_transactions['value_valuedate'] >= pd.Timestamp.now(tz='UTC')]))
df_future_value

In [164]:
df_future_value['ruleset']='11.2.2'
df_future_value

In [166]:
frames = [df_end_before_start_planned, df_end_before_start_actual, df_future_start, df_future_end,df_future_trans, df_future_value]
df_ruleset_fails = pd.concat(frames)

In [167]:
df_ruleset_fails.info()

In [168]:
df_ruleset_fails = df_ruleset_fails[['iatiidentifier', "reportingorg_ref", "reportingorg_narrative",'ruleset']]
df_ruleset_fails

In [169]:
## Rename the ruleset category to errorcode and cast it to Category datatype
df_ruleset_fails = df_ruleset_fails.rename(columns={"ruleset": "errorcode"})
df_ruleset_fails['errorcode'] = df_ruleset_fails['errorcode'].astype('category')

## Calculate some Big Number variables from df_ruleset_fails
total_ruleset_fails = len(df_ruleset_fails)
total_ruleset_publishers = df_ruleset_fails["reportingorg_narrative"].nunique()
total_ruleset_activities = df_ruleset_fails["iatiidentifier"].nunique()

print(total_ruleset_fails)
print(total_ruleset_publishers)
print(total_ruleset_activities)

In [170]:
df_ruleset_fails.info()

In [171]:
df_ruleset_fails_byrule = (
        df_ruleset_fails.groupby("errorcode").size()
        .rename("Failures").reset_index()
        .sort_values(["Failures","errorcode"], ascending=[False, True])
    )
df_ruleset_fails_byrule

In [172]:
df_ruleset_fails_byorg = pd.DataFrame(df_ruleset_fails.groupby(['reportingorg_narrative', 'errorcode']).size().reset_index())
df_ruleset_fails_byorg = df_ruleset_fails_byorg.rename({0: 'number_of_failures'}, axis='columns')
df_ruleset_fails_byorg

### How recently have organisations been publishing?
Find the most recent activity that a publisher has updated. So, instead of looking at all activities, we now focus on publishers

In [189]:
## Need to investigate using a max-type of function rather than sorting and taking the 'tail'
df_org_yearlastpublished = df_lastupdated.sort_values('Year').groupby('reportingorg_ref').tail(1)
df_org_yearlastpublished.info()

In [190]:
df_org_yearlastpublished_group = df_org_yearlastpublished.groupby('Year').size().sort_values(ascending = False).reset_index()
df_org_yearlastpublished_group = df_org_yearlastpublished_group.rename({0: 'count'}, axis='columns')
df_org_yearlastpublished_group

In [191]:
## Create a DF of defunct orgs who haven't published since at least 2023, based on lastupdateddatetime
df_defunct_orgs = df_org_yearlastpublished.loc[(df_org_yearlastpublished['Year'] <= 2023)]
df_defunct_orgs

In [192]:
# Calculate number of orgs whose most recent lastupdateddatetime is before 2024
# Fix: Compare datetime with datetime, not int
lastupdated_orgs_before_2024 = df_org_yearlastpublished[df_org_yearlastpublished['lastupdateddatetime'] < pd.Timestamp('2024-01-01', tz='UTC')].shape[0]
print(lastupdated_orgs_before_2024)

In [193]:
# Create a DataFrame of defunct orgs who haven't published in at least two years, based on lastupdateddatetime
df_defunct_orgs = df_org_yearlastpublished.loc[(df_org_yearlastpublished['lastupdateddatetime'] <= (pd.to_datetime('today').normalize() - pd.Timedelta(days=730)).tz_localize('UTC'))]
df_defunct_orgs

In [194]:
df_defunct_orgs.groupby('reportingorg_type').size().sort_values(ascending = False).reset_index()

### Transaction Span

Select data from full df

In [200]:
#select columns
all_trans = df_ds_transactions[['prefix','reportingorg_ref','transactiondate_isodate','value_usd','transactiontype_code']]

#filter to disbursments and expenditures
filt_trans = all_trans[all_trans['transactiontype_code'].isin(['3','4'])].copy()
filt_trans['Year'] = filt_trans['transactiondate_isodate'].dt.year
filt_trans = filt_trans.drop(['transactiondate_isodate'], axis=1)

Count transactions by org and year

In [201]:
trans_years = filt_trans.groupby(['prefix','reportingorg_ref','Year'],observed=True)['value_usd'].count().reset_index(name='Count transaction')
trans_years

Format to get first and last year

In [202]:
#find first and last transaction year (disb and exp)
min_max = trans_years.groupby('reportingorg_ref',observed=True).agg({'Year': ['min', 'max']}).reset_index()
min_max.columns = min_max.columns.to_series().apply(''.join)

#add on org names, format
trans_min_max = registry[['Publisher','IATI Organisation Identifier','Organization Type']].merge(min_max, left_on='IATI Organisation Identifier', right_on='reportingorg_ref', how='left')
trans_min_max = trans_min_max.drop(['reportingorg_ref'], axis=1)
trans_min_max = trans_min_max.dropna()
trans_min_max  = trans_min_max.rename(columns={'Yearmin': 'First transaction Year','Yearmax': 'Last transaction Year'})
trans_min_max = trans_min_max.reset_index(drop=True)
trans_min_max

### Budget Span

Select data from full df

In [203]:
#select columns
all_budgs = df_ds_budget[['prefix','reportingorg_ref','periodstart_isodate','_link']].copy()

#filter to disbursments and expenditures
all_budgs['Year'] = all_budgs['periodstart_isodate'].dt.year
all_budgs = all_budgs.drop(['periodstart_isodate'], axis=1)

Count of budgets by org and year

In [204]:
budg_years = all_budgs.groupby(['prefix','reportingorg_ref','Year'],observed=True)['_link'].count().reset_index(name='Count')
budg_years

Format to get first and last budget year

In [205]:
#,find first and last budget year 
min_max = budg_years.groupby('reportingorg_ref',observed=True).agg({'Year': ['min', 'max']}).reset_index()
min_max.columns = min_max.columns.to_series().apply(''.join)

#add on org names, format
budg_min_max = registry[['Publisher','IATI Organisation Identifier','Organization Type']].merge(min_max, left_on='IATI Organisation Identifier', right_on='reportingorg_ref', how='left')
budg_min_max = budg_min_max.drop(['reportingorg_ref'], axis=1)
budg_min_max = budg_min_max.dropna()
budg_min_max  = budg_min_max.rename(columns={'Yearmin': 'First budget start Year','Yearmax': 'Last budget start Year'})
budg_min_max = budg_min_max.reset_index(drop=True)
budg_min_max

### Organisation level data

# This section summarises activities by reporting organisation and activity status,
# counts the number of activities per status, enriches the results with organisation names,
# and sorts the output to highlight the most common statuses for each organisation.


In [206]:
# Group activities by reporting organisation and activity status
# and count how many activities fall into each group

df_timeliness_by_statustype = (
    df_ds_activities
      .groupby(["reportingorg_ref", "activitystatus_codename"])
      .size()
      .reset_index(name="Number of Activities")
)

In [207]:
# Merge organisation names into the activity status summary
# to make the output more readable and informative

df_timeliness_by_statustype = df_timeliness_by_statustype.merge(
    df_summary[["reportingorg_ref", "reportingorg_name"]].drop_duplicates(),
    on="reportingorg_ref",
    how="left"
)

In [208]:
# Sort results by reporting organisation and
# then by number of activities (highest first)

df_timeliness_by_statustype = df_timeliness_by_statustype.sort_values(
    ["reportingorg_ref", "Number of Activities"],
    ascending=[True, False]
)

## Store results

In [209]:
## Fill empty values
df_summary['total_activities'] = df_summary['total_activities'].fillna(0)
df_summary['_flag'] = df_summary['_flag'].fillna("green")

In [210]:
df_big_numbers = pd.DataFrame({
    'total': ['unique_activities', 'unique_activities_orgs',  'total_implementing_activities', 'total_implementing_orgs', 'unique_implementing_activities_noenddate', 'unique_implementing_orgs_noenddate', 'unique_implementing_activities_dashboard', 'unique_implementing_orgs_dashboard','mean_activities', 'median_activities', 'mode_activities', 'max_activities', 'org_with_max_activities', 'unique_implementation_ended', 'unique_closed_notended', 'lastupdated_nulls', 'lastupdated_old', 'lastupdated_nonnulls', 'lastupdated_iati_era', 'finalisation_actualend', 'finalisation_notended', 'zerodifforgs', 'zerodiffactivities', 'total_ruleset_fails', 'total_ruleset_publishers', 'total_ruleset_activities', 'unique_act_dates', 'lastupdated_acts_before_2024', 'lastupdated_orgs_before_2024', 'last_run_time'],
    'count': [unique_activities, unique_activities_orgs, unique_implementing_activities, unique_implementing_orgs, unique_implementing_activities_noenddate, unique_implementing_orgs_noenddate, unique_implementing_activities_dashboard, unique_implementing_orgs_dashboard, mean_activities, median_activities, mode_activities, max_activities, org_with_max_activities, unique_implementation_ended, unique_closed_notended, lastupdated_nulls, lastupdated_old, lastupdated_nonnulls, lastupdated_iati_era,finalisation_actualend, finalisation_notended, zerodifforgs, zerodiffactivities, total_ruleset_fails, total_ruleset_publishers, total_ruleset_activities, unique_act_dates, lastupdated_acts_before_2024, lastupdated_orgs_before_2024, last_run_time]
})
df_big_numbers

In [211]:
df_summary

# This section filters activities in Implementation status with a recorded actual end date,
# extracts the year the activity ended, and aggregates counts of completed implementations by reporting organisation and year.

In [212]:
# Create a working copy of the activities dataset
df = df_ds_activities.copy()

# Cleaning to ensure consistent comparison
df["activitystatus_codename"] = df["activitystatus_codename"].astype(str).str.strip()

# Filter for activities in Implementation status with a non-null actual end date
mask = (
    df["activitystatus_codename"].eq("Implementation") &
    df["actualend"].notna()
)

df_impl = df[mask].copy() 
print("Rows after Implementation + actualend filter:", len(df_impl))

# Extract year from actualend into "Actual End Date"
df_impl["Actual End Date"] = df_impl["actualend"].dt.year
df_impl = df_impl.dropna(subset=["Actual End Date"])
df_impl["Actual End Date"] = df_impl["Actual End Date"].astype(int)

# Group by org + year
df_implementation_ended_byyear_byorg = (
    df_impl
    .groupby(["reportingorg_ref", "Actual End Date"], observed=True)
    .size()
    .rename("Number of Activities")
    .reset_index()
)

# Preview the resulting aggregated dataset
print(df_implementation_ended_byyear_byorg.head())

In [213]:
df_implementation_ended_byyear_byorg

In [217]:
df_big_numbers.to_csv('/work/data/big_numbers.csv')

In [213]:
"""
# Write dataframes to file
df_big_numbers.to_csv('/work/data/big_numbers.csv')
df_summary.to_csv('/work/data/summary.csv')
df_ds_activities.to_csv('/work/data/activities.csv')
df_grouped_activities.to_csv("/work/data/grouped_activities.csv")
df_grouped_stacked_activities.to_csv("/work/data/grouped_stacked_activities.csv")
df_implementation_ended.to_csv('/work/data/impended.csv')
df_implementation_ended_byyear.to_csv('/work/data/impended_y.csv')
df_closed_notended_byyear.to_csv('/work/data/closednotended_y.csv')
df_finalisation_actualend_byyear.to_csv('/work/data/finalisation_actualend_byyear.csv')
df_finalisation_notended_byyear.to_csv('/work/data/finalisation_notended_byyear.csv')
df_lastupdated.to_csv('/work/data/lastupdated.csv')
df_lastupdated_byyear.to_csv('/work/data/lastupdated_y.csv')
df_lastupdateddiff.to_csv('/work/data/lastupdateddiff.csv')
df_lastupdated_orgs_byyear.to_csv('/work/data/lastupdated_orgs_byyear.csv')
df_org_yearlastpublished_group.to_csv('/work/data/org_yearlastpublished_group.csv')
df_ruleset_fails.to_csv("/work/data/ruleset_fails.csv")
df_ruleset_fails_byorg.to_csv("/work/data/ruleset_fails_byorg.csv")
df_timeliness_by_orgtype.to_csv("/work/data/timeliness_by_orgtype.csv")
df_timeliness_aggregate_activities.to_csv("/work/data/timeliness_aggregate_activities.csv")
df_timelag_by_orgtype.to_csv("/work/data/timelag_by_orgtype.csv")
df_timelag_aggregate_activities.to_csv("/work/data/timelag_aggregate_activities.csv")
df_timeliness_by_statustype.to_csv("/work/data/timeliness_by_statustype.csv")
df_updated_timeliness_by_orgtype.to_csv('/work/data/updated_timeliness_by_orgtype.csv')
df_updated_timeliness_aggregate_activities.to_csv('/work/data/updated_timeliness_aggregate_activities.csv')
df_ruleset_fails_byrule.to_csv("/work/data/ruleset_fails_byrule.csv")
df_implementation_ended_byyear_byorg.to_csv("/work/data/implementation_ended_byyear_byorg")
week_timeliness.to_csv('/work/data/week_timeliness.csv',index=False)
activity_dates_joined.to_csv('/work/data/activity_dates_joined.csv',index=False)
trans_min_max.to_csv('/work/data/trans_min_max.csv',index=False)
budg_min_max.to_csv('/work/data/budg_min_max.csv',index=False)


In [4]:
## AI agent suggested method for checking hard drive

import subprocess
import shutil

# Method 1: Using shutil (cross-platform, Python-native)
print("=== Disk Space Information (using shutil) ===\n")
try:
    total, used, free = shutil.disk_usage("/")
    
    # Convert bytes to GB for readability
    total_gb = total / (1024**3)
    used_gb = used / (1024**3)
    free_gb = free / (1024**3)
    percent_used = (used / total) * 100
    
    print(f"Total Space: {total_gb:.2f} GB")
    print(f"Used Space:  {used_gb:.2f} GB ({percent_used:.1f}%)")
    print(f"Free Space:  {free_gb:.2f} GB")
except Exception as e:
    print(f"Error getting disk space: {e}")

# Method 2: Using df command (Linux/Unix)
print("\n=== Disk Space Information (using df command) ===\n")
try:
    result = subprocess.run(['df', '-h'], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)
except FileNotFoundError:
    print("df command not found. This command may not be available in this environment.")
except Exception as e:
    print(f"Error running df command: {e}")

# Check specific /work directory if it exists
print("\n=== /work Directory Space ===\n")
try:
    total, used, free = shutil.disk_usage("/work")
    total_gb = total / (1024**3)
    used_gb = used / (1024**3)
    free_gb = free / (1024**3)
    percent_used = (used / total) * 100
    
    print(f"Total Space: {total_gb:.2f} GB")
    print(f"Used Space:  {used_gb:.2f} GB ({percent_used:.1f}%)")
    print(f"Free Space:  {free_gb:.2f} GB")
except Exception as e:
    print(f"/work directory not accessible: {e}")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=f9058ed8-51a0-4f04-bd96-e60901d82949' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>